### Lab Assignment: Python Warmup and Logfile Analytics

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: January 23, 2026

---

This lab consists of two parts:

- Part 1 is the Python warmup
- Part 2 is the logfile analytics in PySpark

Answer the questions in this assignment, showing all code and solutions.

**Total points: 20**

---

### Part 1: Python Warmup

1) (1 PT) Rename this notebook to JupyterTutorial_[your_initials], where you will enter your initials in place of [your_initials].

2) (1 PT) In the cell below, enter a list of data science topics you find interesting.  Use the markdown style (you will need to change the style from the Code style).

- neural networks
- graphs
- survival analysis

3) (1 PT) In the cell below, enter the following Python list:  

some_vals = [1, 6, 10, 44]  

You will use the Code style, run the cell, and print the list.

In [1]:
some_vals = [1,6,10,44]
print(some_vals)

[1, 6, 10, 44]


4) (1 PT) Use a list comprehension to return a filtered list containing only the values greater than 6.  
Call this list *some_vals_filtered* and print it.

In [2]:
[i for i in some_vals if i > 6]

[10, 44]

Next, a small pandas dataframe is constructed.

In [3]:
import pandas as pd

df = pd.DataFrame({'first_name': ['Andy','Crystal'],
                   'domain_facebook' : [1,1],
                   'domain_foursquare' : [0,0],
                   'age' : [20, 32]})
df

,first_name,domain_facebook,domain_foursquare,age
0,Andy,1,0,20
1,Crystal,1,0,32


5) (1 PT) In the cell below, write a list comprehension that returns the fields names in the dataframe `df` containing the string *domain*.  Run the cell to verify the correct result.

In [4]:
[c for c in df.columns if 'domain' in c]

['domain_facebook', 'domain_foursquare']

6) (1 PT) Use the list comprehension from (5) to index into `df` and show the data for columns containing *domain*

In [5]:
df[[c for c in df.columns if 'domain' in c]]

,domain_facebook,domain_foursquare
0,1,0
1,1,0


7) (1 PT) In the cell below, print the *domain_facebook* column

In [6]:
df['domain_facebook']

0    1
1    1
Name: domain_facebook, dtype: int64

8) (1 PT) In the cell below, print the row with index 1.

In [7]:
df.iloc[1,:]

first_name           Crystal
domain_facebook            1
domain_foursquare          0
age                       32
Name: 1, dtype: object

9) (1 PT) Next, you will cube the *age* column of `df` and assign the result to a new column called *agecube*.

Specifically, call the `apply` method with a `lambda function` inside to cube the *age* column.  
Print the dataframe.

In [8]:
df['agecube'] = df.age.apply(lambda x: pow(x,3))
df

,first_name,domain_facebook,domain_foursquare,age,agecube
0,Andy,1,0,20,8000
1,Crystal,1,0,32,32768


10) (1 PT) Given the list of strings below, form one string, placing semicolons between each word.  It should look like this:  

`'the;quick;brown;fox'`

Print the resulting string.

In [9]:
some_list = ['the','quick','brown','fox']
";".join(some_list)

'the;quick;brown;fox'

---

### Part 2: Logfile Analytics

Import modules for Spark Session and regex 

Note: regexes can be used to search strings for patterns. Here is a [reference](https://realpython.com/regex-python/?utm_source=chatgpt.com).

In [10]:
from pyspark.sql import SparkSession
import re

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 12:52:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


11) (1 PT) Read in the logfile.txt data

In [11]:
logs = sc.textFile('logfile.txt')

12) (1 PT) Count the number of rows of data

In [12]:
logs.count()

100000

13) (2 PTS) Show the first five lines containing WARN.  
    Write code to count and print the total number of lines containing WARN.

In [13]:
warn_lines = logs.filter(lambda x: "WARN" in x)
warn_lines.count()

14823

14) (2 PTS) Write a word count program to count the number of each of these log levels:  

- WARN
- INFO
- DEBUG
- ERROR

In [55]:
import re

m = re.compile(r'\[(\w+)\]')

r = logs.map(lambda x: (m.search(x).group(1),1)) \
        .reduceByKey(lambda x,y: x+y)
r.collect()


[('INFO', 70083), ('ERROR', 9951), ('DEBUG', 5143), ('WARN', 14823)]

15. (2 PTS) Return the three log lines with the highest latency. This is reported in the log as `latency_ms`.

Note: There may be more than three lines tied for highest latency, in which case, just show three records.

In [75]:
m = re.compile(r'latency_ms=(\d+)')

r = logs.map(lambda x: (int(m.search(x).group(1)),x) if m.search(x) else (None, x)) \
        .sortByKey(ascending=False) \
        .map(lambda x: x[1])
r.take(3)

['2026-01-01T08:30:03.360000 [INFO] api: heartbeat ip=154.9.232.228 latency_ms=5000 trace=e655i38fjysr seq=3785',
 '2026-01-01T08:37:27.714000 [INFO] metrics: request received ip=220.121.117.138 latency_ms=5000 trace=vtvn3oxagjp3 seq=4736',
 '2026-01-01T08:40:32.661000 [INFO] gateway: connection closed ip=85.231.101.84 latency_ms=5000 trace=3ml5tdora8on seq=5120']

16. (2 PTS) Compute the average latency for each service. Ignore log entries without a latency_ms.

In [74]:
import re

m1 = re.compile(r'\[(\w+)\]')
m2 = re.compile(r'latency_ms=(\d+)')

r = logs.map(lambda x: (m1.search(x).group(1), \
                        (int(m2.search(x).group(1)),1) if m2.search(x) else None )) \
        .reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1])) \
        .map(lambda x: (x[0], x[1][0]/x[1][1]))
r.take(10)


[('INFO', 2499.235035600645),
 ('ERROR', 2506.5652698221284),
 ('DEBUG', 2501.758506708147),
 ('WARN', 2487.169466369831)]